## Instalando bibliotecas

In [1]:
!pip install transformers torch accelerate pdfplumber tqdm -q

## Importando bibliotecas

In [2]:
import pdfplumber
import json
import torch
from transformers import pipeline, logging
from tqdm import tqdm

# Desativa avisos (apenas erros críticos serão mostrados)
logging.set_verbosity_error()

/home/jessica/Área de trabalho/topicosAvancadosEmIAAAvaliacao02/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Função para extrair texto do pdf usando o pdfplumber

In [3]:
def extract_text_pdf(file_path):

    if file_path.lower().endswith('manual_genie.pdf'):
        text = ""
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        return text


## Divisão em trechos (chunks)

In [4]:
def split_text(text, max_chunk_length=500):

    paragraphs = text.split("\n")
    chunks = []
    current_chunk = ""

    for para in paragraphs:
        if len(current_chunk) + len(para) < max_chunk_length:
            current_chunk += para + "\n"
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = para + "\n"

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

#print(split_text(extract_text_pdf("manual_genie.pdf")))

## Gerando perguntas e respostas

In [5]:
def generate_instruction_response(chunk, hf_pipeline):

    prompt = f"""
                You are a data generator for fine-tuning or training a language model.

                #Given the content below, create:
                
                1.The question must be answerable only from the provided content.
                2.Do not include explanations, opinions, or additional information.
                3.If the content contains technical specifications, procedures, safety rules, limits, warnings, controls, or maintenance instructions, prioritize these topics.
                4.Keep the question (instruction) under 15 words, based only on the content.
                5.Keep the answer (resposta) under 20 words, based only on the content.

                Do not add extra explanations. Use the following format:

                INSTRUCTION: <short question>
                RESPONSE: <short answer>

                Content:
                \"\"\"
                {chunk}
                \"\"\"
                """

    messages = [{"role": "user", "content": prompt}]

    try:
        outputs = hf_pipeline(
            messages,
            max_new_tokens=150,
            max_length=None,  # evita warning
            return_full_text=False
        )
        content = outputs[0]["generated_text"]

        # Extrai os campos esperados
        instr_part = content.split("INSTRUCTION:")[1].split("RESPONSE:")[0].strip()
        answer = content.split("RESPONSE:")[1].strip()
        return instr_part, answer
    except Exception as e:
        # Em caso de erro, retorna None, None (o par será ignorado)
        return None, None


## Salvar dataset em formato JSONL

In [6]:
def save_to_jsonl(pairs, output_file="dataset.jsonl"):

    with open(output_file, "w", encoding="utf-8") as f:
        for instruction, answer in pairs:
            if instruction and answer:
                example = {
                    "Instruction": instruction,
                    "Output": answer
                }
                f.write(json.dumps(example, ensure_ascii=False) + "\n")

## Gerando o dataset

In [ ]:
def generate_dataset(file_path="manual_genie.pdf", model_id="...", output_file="dataset.jsonl", max_chunks=None):
    """
    file_path: caminho para o arquivo .pdf ou .txt
    model_id: identificador do modelo no Hugging Face Hub
    output_file: nome do arquivo de saída (JSONL)
    max_chunks: se especificado, limita a quantidade de chunks processados
    """
    print(f"🔄 Carregando modelo: {model_id} ...")

    # Pipeline para geração de texto
    hf_pipeline = pipeline(
        "text-generation",
        model=model_id,
        device_map="auto",        # usa GPU se disponível
    )

    print("📄 Extraindo texto do arquivo...")
    text = extract_text_pdf(file_path)

    print("✂️  Dividindo em chunks...")
    chunks = split_text(text)

    if max_chunks:
        chunks = chunks[:max_chunks]

    print(f"🧠 Gerando pares (instrução + resposta) para {len(chunks)} chunks...")
    pairs = []

    for chunk in tqdm(chunks, desc="Processando chunks"):
        if len(chunk.strip()) < 10:  # ignora chunks muito curtos
            continue
        instruction, answer = generate_instruction_response(chunk, hf_pipeline)
        if instruction and answer:
            pairs.append((instruction, answer))

    save_to_jsonl(pairs, output_file)
    print(f"\n✅ Dataset salvo em: {output_file} ({len(pairs)} exemplos gerados)")


In [ ]:
# Gerando dataset
caminho_arquivo = "manual_genie.pdf" 

generate_dataset(
    file_path=caminho_arquivo,
    model_id="", #mudar o modelo para um modelo robusto com 1M paramtros somente para testar
    output_file="dataset_gerado.jsonl",
    max_chunks=10   # remova esta linha para processar todos os chunks
)

🔄 Carregando modelo: user-anto/Axiom-Dense-380M-Instruct ...


ImportError: This modeling file requires the following packages that were not found in your environment: config, configuration_axiom, model. Run `pip install config configuration_axiom model`